# Phase 1 (KSE-100) — Pooled XGBoost Training

**Standalone notebook.** This notebook has no dependency on any particular codebase — it only needs the Parquet file below and the packages in the cell after this one. Copy this notebook and the `.parquet` file anywhere and it will run.

**Requirements:** `pandas`, `numpy`, `scikit-learn`, `xgboost`, `matplotlib`, `seaborn`, `statsmodels`, `pyarrow`. Install with:
```
pip install pandas numpy scikit-learn xgboost matplotlib seaborn statsmodels pyarrow
```

**The dataset:** a pooled, cross-sectional, next-20-trading-day-return dataset across 97 KSE-100 tickers, already cleaned and feature-selected — bad-tick outliers removed, and every feature column that's identical across all tickers on a given date dropped (a pooled model can't use those to explain *why one ticker moves differently from another*, only to spuriously memorize dates). If you don't have `phase1_model_input_horizon20.parquet` yet, it's produced by a separate data-prep script; ask for that if you need to regenerate it — this notebook only consumes the finished file.

**Why XGBoost, why this horizon:** across a same-methodology comparison of 9 model families on two individual tickers (PSO, MEBL), XGBoost was the most consistent tree ensemble — no blow-ups the way SVR(RBF) or Linear Regression/LSTM had. Next-day return on this dataset turned out to carry essentially no learnable signal beyond a naive "no change" baseline; a horizon sweep (1 / 5 / 20 trading days) found the signal only becomes real at ~20 days out: ~58% directional accuracy vs. a ~46% naive baseline, holding up on 89 of 97 individual tickers. Raw price-magnitude error (MAE/RMSE/MAPE) never clearly beat naive at any horizon tried, including after a volatility-normalized target reformulation — treat this model as a **directional signal**, not a magnitude/price forecaster. Section 1b below exists specifically to show *why*, quantitatively and visually, before any model gets trained.

**What this notebook does:**
1. Loads the prepared Parquet dataset (Section 1).
2. Runs an extensive exploratory diagnostic pass on the raw feature/target relationship *before* touching a model — linear and nonlinear (mutual information) signal strength, train/test regime drift, zero-inflation, per-ticker coverage, sector structure, and own-history autocorrelation — so "the model isn't training" is understood as a data property, not a training bug (Section 1b).
3. Reconstructs the same chronological train / validation / test split used when the dataset was built (Section 2).
4. Trains XGBoost with a held-out validation slice and early stopping, so you can watch it overfit/underfit round by round (Section 3).
5. Evaluates with price-level MAE/RMSE/MAPE and directional accuracy against a naive persistence baseline, plus an after-the-fact overfit/underfit verdict (Section 4).
6. Reports per-ticker breakdown and feature importances (Sections 5-6).
7. Optionally saves the trained model (Section 7).

Run the cells top to bottom.

## 1. Setup & load the dataset

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_error as mae_fn,
    mean_squared_error as mse_fn,
    mean_absolute_percentage_error as mape_fn,
)
from sklearn.feature_selection import mutual_info_regression
from xgboost import XGBRegressor

sns.set_theme(style='whitegrid')

# Point this at wherever you've placed the Parquet file - it does not need to sit next
# to this notebook or inside any particular project structure.
PARQUET_PATH = 'phase1_model_input_horizon20.parquet'

df = pd.read_parquet(PARQUET_PATH)
df['date'] = pd.to_datetime(df['date'])

# The target column is named target_return_t{HORIZON} - infer HORIZON from whichever
# one is actually in the file, so this notebook works unmodified against a re-export
# built at a different horizon.
target_col = next(c for c in df.columns if c.startswith('target_return_t'))
HORIZON = int(target_col.replace('target_return_t', ''))

print(f"Loaded {df.shape[0]} rows x {df.shape[1]} columns from {PARQUET_PATH}")
print(f"Target: {target_col}  (HORIZON={HORIZON} trading days)")
print(f"Tickers: {df['ticker'].nunique()}   Date range: {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"Split counts: {df['split'].value_counts().to_dict()}")

## 1b. Exploratory Data Analysis — diagnosing why training struggles

If the model "isn't training" (early stopping fires almost immediately, or MAE never clearly beats naive), the cause is usually visible in the data *before* a model ever sees it. Everything in this section reads `df` only — nothing here is model output, and nothing here mutates `df` in place (each cell works off `df.copy()` or a local subset), so it's safe to run before or after Section 2.

### 1b.1 Target distribution & regime drift over time

A target that's dominated by noise (near-zero mean, fat tails, no visible structure) is the first sign a model will struggle. The by-year panel checks something different: whether the *relationship* the model has to learn has been drifting over the training history — if `mean` swings from strongly negative to strongly positive across years, a model trained on the full history is being asked to fit a moving target, and no amount of tuning fixes that on its own (this is exactly what caused a classification-reframing experiment to fail on this dataset in earlier work — it's a real, recurring issue here, not a one-off).

In [ ]:
print(df[target_col].describe())
print(f"\nSkew: {df[target_col].skew():.3f}   Kurtosis: {df[target_col].kurt():.3f}")
print("(Kurtosis well above 0 means fat tails vs. a normal distribution - a few extreme "
      "moves dominate squared-error metrics like RMSE; skew far from 0 means the up/down "
      "moves aren't symmetric.)")

eda_df = df.copy()
eda_df['year'] = eda_df['date'].dt.year
yearly = eda_df.groupby('year')[target_col].agg(['mean', 'std', 'count'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df[target_col], bins=100, color='steelblue', ax=axes[0])
axes[0].axvline(0, color='crimson', linestyle='--')
axes[0].set_title(f'{target_col} distribution (all rows)')

yearly['mean'].plot(ax=axes[1], marker='o', color='darkorange')
axes[1].axhline(0, color='gray', linestyle=':')
axes[1].set_title(f'{target_col} mean by year (regime drift check)')
axes[1].set_ylabel('mean target return')
plt.tight_layout()
plt.show()

display(yearly)

### 1b.2 Target distribution by year — full shape, not just the mean

A violin plot shows what the year-by-year `mean` table above can hide: whether a year's average was pulled around by a fat tail, and how the *spread* (not just the center) has changed. Widening violins over time mean later years are noisier, not just differently-directioned — a second, independent reason a fixed model can struggle to generalize forward.

In [ ]:
plot_sample = eda_df.sample(n=min(60000, len(eda_df)), random_state=42)

fig, ax = plt.subplots(figsize=(15, 5))
sns.violinplot(x='year', y=target_col, data=plot_sample, ax=ax, inner='quartile', cut=0, density_norm='width')
ax.axhline(0, color='gray', linestyle=':')
ax.set_ylim(df[target_col].quantile(0.01), df[target_col].quantile(0.99))
ax.set_title(f'{target_col} distribution by year (60k-row sample; y-axis clipped to 1st-99th percentile)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 1b.3 Feature-target correlation ranking (linear signal)

The most direct linear diagnostic: how strongly does *any* individual feature relate to the target, on its own? If the **maximum** |correlation| across all ~140 features is tiny (well under 0.05-0.10), that's quantitative evidence there isn't much *linear* signal for a model to find. It doesn't rule out nonlinear signal, though — that's what 1b.4 (mutual information) checks next.

In [ ]:
non_feature_cols = ['ticker', 'sector', 'date', 'split', 'close', target_col]
numeric_feature_cols = [
    c for c in df.columns
    if c not in non_feature_cols and pd.api.types.is_numeric_dtype(df[c])
]
print(f"{len(numeric_feature_cols)} numeric feature columns")

corrs = df[numeric_feature_cols + [target_col]].corr(numeric_only=True)[target_col].drop(target_col)
corrs_sorted = corrs.reindex(corrs.abs().sort_values(ascending=False).index)

print(f"\nMax |correlation| across all {len(corrs)} features: {corrs.abs().max():.4f}")
print(f"Median |correlation|: {corrs.abs().median():.4f}")
print(f"Features with |correlation| > 0.05: {(corrs.abs() > 0.05).sum()} / {len(corrs)}")

top20_corr = corrs_sorted.head(20)
fig, ax = plt.subplots(figsize=(8, 8))
colors = ['crimson' if v < 0 else 'steelblue' for v in top20_corr.values]
sns.barplot(x=top20_corr.values, y=top20_corr.index, hue=top20_corr.index,
            palette=colors, legend=False, ax=ax)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 20 features by Pearson correlation with target')
ax.set_xlabel('Pearson correlation')
plt.tight_layout()
plt.show()

### 1b.4 Mutual information ranking (nonlinear signal)

Pearson correlation only sees straight-line relationships. Mutual information (computed here via a k-NN estimator on a 30k-row sample, for speed) picks up **any** statistical dependency, linear or not — the thing that actually matters for a tree model like XGBoost, which splits on thresholds rather than fitting a line. Comparing this ranking to 1b.3's: if the same handful of features top both lists, that's real, consistent signal a tree could plausibly exploit. If MI surfaces *different* top features than correlation did, that's a nonlinear relationship worth inspecting directly in 1b.5's scatter plots — and if MI scores are uniformly tiny too, that's the strongest evidence yet that the ceiling is a property of the data, not of any one method's blind spot.

In [ ]:
mi_sample = df.sample(n=min(30000, len(df)), random_state=42)
X_mi = mi_sample[numeric_feature_cols].fillna(0.0)
y_mi = mi_sample[target_col]

mi_scores = mutual_info_regression(X_mi, y_mi, random_state=42, n_neighbors=5)
mi_series = pd.Series(mi_scores, index=numeric_feature_cols).sort_values(ascending=False)

print(f"Max MI score: {mi_series.max():.4f}   Median: {mi_series.median():.4f}")
n_overlap = len(set(mi_series.head(20).index) & set(corrs_sorted.head(20).index))
print(f"Overlap between top-20 by MI and top-20 by Pearson correlation: {n_overlap} / 20")

fig, ax = plt.subplots(figsize=(8, 8))
top20_mi = mi_series.head(20)
sns.barplot(x=top20_mi.values, y=top20_mi.index, color='mediumseagreen', ax=ax)
ax.set_title('Top 20 features by mutual information with target\n(captures nonlinear signal Pearson correlation misses)')
ax.set_xlabel('Mutual information')
plt.tight_layout()
plt.show()

### 1b.5 What the strongest relationships actually look like

Numbers can hide shape. These are scatter plots (with a LOWESS trend line) of the target against the union of the top-3 Pearson-correlated and top-3 MI-ranked features, on a 15k-row sample for readability. Look for: a visible slope or curve in the trend line (real, if weak, structure) vs. a flat line through a shapeless cloud (no usable relationship at any single-feature level, regardless of what the correlation/MI number technically says).

In [ ]:
top_scatter_feats = list(dict.fromkeys(
    list(corrs_sorted.head(3).index) + list(mi_series.head(3).index)
))[:6]
scatter_sample = df.sample(n=min(15000, len(df)), random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feat in zip(axes.flat, top_scatter_feats):
    sns.regplot(
        x=feat, y=target_col, data=scatter_sample, ax=ax,
        scatter_kws={'alpha': 0.15, 's': 8}, line_kws={'color': 'crimson'}, lowess=True,
    )
    ax.set_title(feat)
    ax.axhline(0, color='gray', linewidth=0.6, linestyle=':')
plt.tight_layout()
plt.show()

### 1b.6 Constant / near-constant / zero-inflated feature audit

During data prep, missing values were filled with `0.0` — for a feature that's frequently or always missing for many tickers, that produces a column that's mostly (or entirely) zero, which a tree model can't split on productively. This flags exactly which columns that's happened to, rather than leaving it as an invisible drag on training.

In [ ]:
nunique = df[numeric_feature_cols].nunique()
constant_cols = nunique[nunique <= 1].index.tolist()

stds = df[numeric_feature_cols].std()
near_zero_var_cols = stds[stds < 1e-6].index.tolist()

zero_pct = (df[numeric_feature_cols] == 0).mean().sort_values(ascending=False)

print(f"{len(constant_cols)} feature columns are literally constant across the whole dataset: {constant_cols}")
print(f"{len(near_zero_var_cols)} feature columns have near-zero variance (std < 1e-6): {near_zero_var_cols}")
print(f"\n{(zero_pct > 0.90).sum()} feature columns are more than 90% exact zeros.")

fig, ax = plt.subplots(figsize=(8, 7))
top20_zero = zero_pct.head(20)
sns.barplot(x=top20_zero.values * 100, y=top20_zero.index, color='goldenrod', ax=ax)
ax.set_title('Top 20 features by fraction of exact-zero values')
ax.set_xlabel('% exact zeros')
plt.tight_layout()
plt.show()

### 1b.7 Train vs. test distribution shift — visualized

The train/test split here is a hard chronological cutoff (train: everything before it, test: everything after) — spanning a ~16-year train window against a ~3.5-year test window. Overlapping density curves make a distributional shift visible directly, rather than relying only on a difference-in-means number: a shift you can see (not just compute) is a shift the model genuinely had no way to prepare for — exactly the failure mode that broke an earlier classification-reframing attempt on this same dataset (fit-period class balance didn't match the test period's).

In [ ]:
train_y = df.loc[df['split'] == 'train', target_col]
test_y = df.loc[df['split'] == 'test', target_col]
print(f"Target mean/std -  train: {train_y.mean():.4f} / {train_y.std():.4f}   "
      f"test: {test_y.mean():.4f} / {test_y.std():.4f}")

top_feats = corrs_sorted.head(6).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
sns.kdeplot(train_y, fill=True, alpha=0.3, label='train', ax=axes[0, 0],
            clip=(train_y.quantile(0.01), train_y.quantile(0.99)))
sns.kdeplot(test_y, fill=True, alpha=0.3, label='test', ax=axes[0, 0],
            clip=(train_y.quantile(0.01), train_y.quantile(0.99)))
axes[0, 0].set_title(target_col)
axes[0, 0].legend()

for ax, feat in zip(axes.flat[1:], top_feats):
    lo, hi = df[feat].quantile([0.01, 0.99])
    sns.kdeplot(df.loc[df['split'] == 'train', feat], fill=True, alpha=0.3, ax=ax, clip=(lo, hi))
    sns.kdeplot(df.loc[df['split'] == 'test', feat], fill=True, alpha=0.3, ax=ax, clip=(lo, hi))
    ax.set_title(feat)
plt.suptitle('Train (blue) vs. test (orange) distributions — target + top 6 correlated features')
plt.tight_layout()
plt.show()

train_stats = df.loc[df['split'] == 'train', top_feats].agg(['mean', 'std'])
test_stats = df.loc[df['split'] == 'test', top_feats].agg(['mean', 'std'])
shift_df = pd.DataFrame({
    'train_mean': train_stats.loc['mean'], 'test_mean': test_stats.loc['mean'],
    'train_std': train_stats.loc['std'],
})
shift_df['shift_in_train_stds'] = (
    (shift_df['test_mean'] - shift_df['train_mean']) / shift_df['train_std'].replace(0, np.nan)
)
display(shift_df.reindex(shift_df['shift_in_train_stds'].abs().sort_values(ascending=False).index))

### 1b.8 Does sector explain any of the cross-sectional spread?

If sector membership systematically shifts the target's distribution, that's real cross-sectional structure the model can exploit through the `sector` feature. If every sector's box looks about the same, sector isn't doing much work for this target.

In [ ]:
sector_order = df.groupby('sector')[target_col].median().sort_values().index

fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(x='sector', y=target_col, data=df, order=sector_order, showfliers=False, ax=ax)
ax.axhline(0, color='gray', linestyle=':')
ax.set_title(f'{target_col} distribution by sector (outliers hidden for readability)')
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

sector_meds = df.groupby('sector')[target_col].median().sort_values(ascending=False)
print(f"Sector median spread: {sector_meds.max():.4f} (best) to {sector_meds.min():.4f} (worst) "
      f"vs. overall median {df[target_col].median():.4f}")

### 1b.9 Does a ticker's own history predict its own future? (autocorrelation)

Independent of any engineered feature: does a stock's own past 20-day returns correlate with its future ones? Positive bars mean momentum (a good period tends to be followed by another good one), negative bars mean mean-reversion, and bars near zero at every lag mean there's no self-predictive structure in the target series itself for that ticker — checked here on the 6 longest-history tickers, where there's enough data for the estimate to mean something.

In [ ]:
row_counts = df.groupby('ticker').size().sort_values()
longest_tickers = row_counts.tail(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, t in zip(axes.flat, longest_tickers):
    series = df.loc[df['ticker'] == t].sort_values('date')[target_col].reset_index(drop=True)
    acf_vals = [series.autocorr(lag=lag) for lag in range(1, 11)]
    sns.barplot(x=list(range(1, 11)), y=acf_vals, color='slateblue', ax=ax)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'{t}  (n={row_counts[t]})')
    ax.set_xlabel('lag (x20 trading days)')
    ax.set_ylim(-0.5, 0.5)
plt.suptitle("Autocorrelation of each ticker's own target series (lags 1-10)")
plt.tight_layout()
plt.show()

### 1b.10 Per-ticker history length

97 tickers pooled together doesn't mean 97 equally-informative tickers — some have only a couple hundred rows after the outlier filter (newer listings, or tickers with lots of filtered bad ticks), diluting the pooled model's ability to learn ticker-specific patterns for them specifically.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(row_counts, bins=30, color='seagreen', ax=ax)
ax.set_title('Rows per ticker (after outlier filtering)')
ax.set_xlabel('Rows')
plt.tight_layout()
plt.show()

print(f"Shortest-history ticker: {row_counts.index[0]} ({row_counts.iloc[0]} rows)")
print(f"Longest-history ticker:  {row_counts.index[-1]} ({row_counts.iloc[-1]} rows)")
print("\n10 shortest-history tickers:")
display(row_counts.head(10).to_frame('rows'))

### 1b.11 Redundancy among the top correlated features

In [ ]:
top15 = corrs_sorted.head(15).index.tolist()
corr_matrix = df[top15].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            square=True, cbar_kws={'label': 'correlation'}, ax=ax)
ax.set_title('Correlation among the top 15 target-correlated features')
plt.tight_layout()
plt.show()

### 1b.12 Reading this section together

Weigh all of 1b.1-1b.11 together before touching Section 3's model:

- **Low max correlation AND low max mutual information (1b.3, 1b.4)** is the headline finding — if nothing clears a small threshold under *either* a linear or nonlinear measure, no amount of hyperparameter tuning manufactures signal that isn't there. Confirm this by checking whether 1b.5's scatter plots show any visible trend line or just a flat line through noise.
- **Regime drift (1b.1, 1b.2) and covariate/target shift (1b.7)** explain *why* a model can look fine on training data and still fail to generalize — not an overfitting bug, a moving-target problem. The KDE overlays make this visible directly, not just as a number.
- **Sector structure (1b.8)** and **own-history autocorrelation (1b.9)** are two independent checks for exploitable structure outside the engineered feature set — if sector boxes and autocorrelation bars both look flat, that's further evidence the ceiling is real.
- **Zero-inflation (1b.6) and redundancy (1b.11)** explain why the *effective* feature count is smaller than the raw column count suggests.
- **Short-history tickers (1b.10)** are a secondary effect — worth knowing about, but rarely the primary cause on their own.

On this dataset specifically, the conclusion already reached (Sections 3-4 reproduce it) is that raw price-magnitude prediction hits a real ceiling here, while a directional signal survives — consistent with a small-but-nonzero correlation/MI ceiling rather than a training bug.

## 2. Reconstruct the train / validation / test split

The `split` column (train/test) is a **chronological** split baked into the dataset already — every train-period row's date precedes every test-period row's date, so there's no lookahead leakage across tickers. A validation slice is carved from the **tail of the training window only** (chronological, last ~15%), used purely to monitor training in Section 3; the test set stays completely unseen until Section 4.

In [ ]:
id_cols = ['ticker', 'sector', 'date', 'split', 'close', target_col]
feature_cols = [c for c in df.columns if c not in id_cols]

train_df = df[df['split'] == 'train'].copy()
test_df = df[df['split'] == 'test'].copy()

val_cutoff_date = train_df['date'].sort_values().iloc[int(len(train_df) * 0.85)]
fit_df = train_df[train_df['date'] < val_cutoff_date]
val_df = train_df[train_df['date'] >= val_cutoff_date]

def make_X(frame, ticker_categories=None, sector_categories=None):
    X = frame[feature_cols].copy()
    X['ticker'] = pd.Categorical(frame['ticker'], categories=ticker_categories)
    X['sector'] = pd.Categorical(frame['sector'], categories=sector_categories)
    return X

# categories are fixed from the fit set, so validation/test rows with a category the
# model never trained on don't silently break the categorical encoding
ticker_categories = fit_df['ticker'].astype('category').cat.categories
sector_categories = fit_df['sector'].astype('category').cat.categories

X_fit = make_X(fit_df, ticker_categories, sector_categories)
X_val = make_X(val_df, ticker_categories, sector_categories)
X_test = make_X(test_df, ticker_categories, sector_categories)
y_fit, y_val, y_test = fit_df[target_col], val_df[target_col], test_df[target_col]
close_fit, close_val, close_test = fit_df['close'], val_df['close'], test_df['close']
ticker_test = test_df['ticker']

# naive baseline: "today's direction persists" - sign(daily_return) if that feature is in
# the dataset, else the sign of the most recent close-over-close move per ticker
if 'daily_return' in df.columns:
    naive_fit_dir = np.sign(fit_df['daily_return'].to_numpy())
    naive_test_dir = np.sign(test_df['daily_return'].to_numpy())
else:
    naive_fit_dir = np.sign(fit_df.groupby('ticker')['close'].diff().fillna(0.0).to_numpy())
    naive_test_dir = np.sign(test_df.groupby('ticker')['close'].diff().fillna(0.0).to_numpy())

print(f"Fit:        {len(X_fit)} rows  ({fit_df['date'].min().date()} -> {fit_df['date'].max().date()})")
print(f"Validation: {len(X_val)} rows  ({val_df['date'].min().date()} -> {val_df['date'].max().date()})")
print(f"Test:       {len(X_test)} rows  ({test_df['date'].min().date()} -> {test_df['date'].max().date()})")

## 3. Train XGBoost — watching it overfit/underfit live, during training

Hyperparameters match the source project's production configuration (`n_estimators=200, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8`). An `eval_set` of `[(fit), (validation)]` makes XGBoost report its error **after every single boosting round**, on both the data it's learning from and a held-out slice it never trains on — the direct way to watch the fit/overfit/underfit transition happen live rather than judging it only after the fact from one final number:

- **Both curves still dropping together** → the model is still learning real signal; not done yet.
- **Both curves flat and high from early on** → **underfitting** — the model isn't finding enough signal even in the data it's fit on.
- **Train keeps dropping but validation flattens or climbs** → **overfitting begins at that round**. `early_stopping_rounds=30` watches for exactly this and halts training at the best validation round instead of running the full budget.

In [ ]:
model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    enable_categorical=True,
    eval_metric='mae',
    early_stopping_rounds=30,
)

print("Training (watch validation_1-mae — that's the live overfit/underfit signal)...")
model.fit(
    X_fit, y_fit,
    eval_set=[(X_fit, y_fit), (X_val, y_val)],
    verbose=25,
)

evals_result = model.evals_result()
best_iter = model.best_iteration
configured_rounds = model.get_params()['n_estimators']
print(f"\nStopped at boosting round {best_iter} of {configured_rounds} configured "
      f"(early stopping picked this as the best validation-MAE round).")
if best_iter < configured_rounds - 1:
    print("Rounds beyond this point were making the fit-subset error go down while "
          "validation error stopped improving - i.e. overfitting had begun.")

### Learning curve — the actual "watch it overfit" plot

Train (fit-subset) MAE vs. validation MAE, one point per boosting round. Read this left-to-right as if training were still running round by round — this is the live view.

In [ ]:
fit_curve = evals_result['validation_0']['mae']
val_curve = evals_result['validation_1']['mae']
rounds = range(1, len(fit_curve) + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rounds, fit_curve, label='Train MAE (fit subset)', color='steelblue')
ax.plot(rounds, val_curve, label='Validation MAE', color='crimson')
ax.axvline(best_iter, color='gray', linestyle='--',
           label=f'Early-stopping pick (round {best_iter})')
ax.set_xlabel('Boosting round')
ax.set_ylabel('MAE (return scale)')
ax.set_title('XGBoost learning curve — train vs. validation, per boosting round')
ax.legend()
plt.tight_layout()
plt.show()

gap_final = val_curve[-1] - fit_curve[-1]
gap_at_best = val_curve[best_iter] - fit_curve[best_iter]
print(f"Train/validation MAE gap at the FINAL round run ({len(fit_curve)}): {gap_final:.5f}")
print(f"Train/validation MAE gap at the early-stopping round ({best_iter}):  {gap_at_best:.5f}")
if gap_final > gap_at_best * 1.3:
    print("\nThe gap widened noticeably after the early-stopping round - a clear live "
          "signature of overfitting setting in past that point.")

## 4. Evaluate — and the post-hoc overfit/underfit verdict

Predictions/targets are reconstructed to price level (`close_at_t * (1 + return)`) before computing MAE/RMSE/MAPE. For directional accuracy, the naive baseline is **"today's direction persists"** — not a naive-zero prediction, which would make directional accuracy look artificially awful (`sign(0)` essentially never matches a real nonzero return).

**Read this alongside the directional accuracy row, not instead of it.** The MAE-based verdict below will likely say "underfitting" — the model doesn't clearly beat naive on raw magnitude, a real and repeatedly-confirmed limitation of this dataset/target (see Section 1b.3/1b.4's correlation and mutual-information ceilings), not a bug — but directional accuracy is the number actually worth trusting here.

In [ ]:
def directional_accuracy(y_true, y_pred):
    """Percentage of correct sign predictions, in [0, 100]."""
    return (np.sign(y_true) == np.sign(y_pred)).mean() * 100


def evaluate_predictions(y_true, y_pred, close_at_t, dir_pred=None):
    """Price-level MAE/RMSE/MAPE. Directional accuracy compares y_true's sign against
    dir_pred's sign if given, else against y_pred's sign (use dir_pred for a naive
    baseline - see the markdown above for why naive-zero can't be used for direction)."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    close_at_t = np.asarray(close_at_t)
    actual_prices = close_at_t * (1 + y_true)
    pred_prices = close_at_t * (1 + y_pred)
    dir_signal = np.asarray(dir_pred) if dir_pred is not None else y_pred

    return {
        'mae': mae_fn(actual_prices, pred_prices),
        'rmse': np.sqrt(mse_fn(actual_prices, pred_prices)),
        'mape': mape_fn(actual_prices, pred_prices) * 100,
        'directional_accuracy': directional_accuracy(y_true, dir_signal),
    }


fit_preds = model.predict(X_fit)
test_preds = model.predict(X_test)

fit_metrics = evaluate_predictions(y_fit, fit_preds, close_fit)
test_metrics = evaluate_predictions(y_test, test_preds, close_test)
naive_fit_metrics = evaluate_predictions(y_fit, np.zeros_like(y_fit), close_fit, dir_pred=naive_fit_dir)
naive_metrics = evaluate_predictions(y_test, np.zeros_like(y_test), close_test, dir_pred=naive_test_dir)

summary = pd.DataFrame(
    [fit_metrics, test_metrics, naive_fit_metrics, naive_metrics],
    index=['XGBoost (fit subset)', 'XGBoost (test)',
           'Naive persistence (fit subset)', 'Naive persistence (test)'],
)
display(summary)


def diagnose(fit_mae, test_mae, naive_fit_mae, naive_test_mae):
    beats_naive_fit = fit_mae < naive_fit_mae * 0.98
    beats_naive_test = test_mae < naive_test_mae * 0.98
    ratio = test_mae / fit_mae
    if not beats_naive_fit and not beats_naive_test:
        return ratio, 'Underfitting - does not clearly beat naive even on the fit subset'
    if ratio > 1.5 and beats_naive_fit:
        return ratio, f'Overfitting - test error is {ratio:.1f}x fit-subset error'
    if ratio > 1.15:
        return ratio, 'Mild overfitting - some generalization gap'
    return ratio, 'Reasonable fit - fit-subset/test performance are close'


ratio, verdict = diagnose(
    fit_metrics['mae'], test_metrics['mae'],
    naive_fit_metrics['mae'], naive_metrics['mae'],
)
print(f"\ntest/fit MAE ratio: {ratio:.2f}x")
print(f"Verdict: {verdict}")

## 5. Per-ticker breakdown (test set)

Same pooled model, evaluated separately on each ticker's test-period rows.

In [ ]:
per_ticker_rows = []
for t in sorted(ticker_test.unique()):
    mask = (ticker_test == t).values
    if mask.sum() < 5:
        continue
    m = evaluate_predictions(
        y_test.values[mask], test_preds[mask], close_test.values[mask]
    )
    m['ticker'] = t
    m['n_test_rows'] = int(mask.sum())
    per_ticker_rows.append(m)

per_ticker_df = pd.DataFrame(per_ticker_rows).set_index('ticker')
per_ticker_df = per_ticker_df[['n_test_rows', 'mae', 'rmse', 'mape', 'directional_accuracy']]
display(per_ticker_df.sort_values('directional_accuracy', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(per_ticker_df['directional_accuracy'], bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(50, color='crimson', linestyle='--', label='coin-flip (50%)')
axes[0].axvline(naive_metrics['directional_accuracy'], color='darkorange', linestyle='--', label='naive (pooled)')
axes[0].set_title('Directional accuracy across tickers')
axes[0].set_xlabel('Directional accuracy (%)')
axes[0].legend()

axes[1].hist(per_ticker_df['mape'], bins=20, color='seagreen', edgecolor='white')
axes[1].set_title('MAPE across tickers')
axes[1].set_xlabel('MAPE (%)')
plt.tight_layout()
plt.show()

n_beat_naive = (per_ticker_df['directional_accuracy'] > naive_metrics['directional_accuracy']).sum()
n_beat_coinflip = (per_ticker_df['directional_accuracy'] > 50).sum()
print(f"Tickers beating naive persistence's directional accuracy: {n_beat_naive}/{len(per_ticker_df)}")
print(f"Tickers beating a coin flip (>50% directional accuracy): {n_beat_coinflip}/{len(per_ticker_df)}")

## 6. Feature importances

In [ ]:
importances = pd.Series(model.feature_importances_, index=X_fit.columns).sort_values(ascending=False)
top_n = 25

fig, ax = plt.subplots(figsize=(8, 9))
importances.head(top_n).iloc[::-1].plot(kind='barh', ax=ax, color='darkslateblue')
ax.set_title(f'XGBoost feature importance — top {top_n} (Phase 1 pooled)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

importances.head(top_n)

## 7. Save the trained model (optional)

In [ ]:
import pickle

out_path = 'phase1_xgboost_pooled.pkl'
with open(out_path, 'wb') as f:
    pickle.dump({
        'model': model,
        'feature_cols': list(X_fit.columns),
        'ticker_categories': list(ticker_categories),
        'sector_categories': list(sector_categories),
        'target_col': target_col,
        'val_cutoff_date': str(val_cutoff_date.date()),
        'best_iteration': best_iter,
        'fit_metrics': fit_metrics,
        'test_metrics': test_metrics,
        'naive_metrics': naive_metrics,
        'overfit_underfit_verdict': verdict,
    }, f)

print(f"Saved: {out_path}")